# 04.5 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/05-scaling/exercises.ipynb)

Run the setup cell first.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"train {len(X_train):,} / test {len(X_test):,}")

spend5 = np.array([200, 400, 600, 800, 1000], dtype=float)

abc = pd.DataFrame({"orders": [2, 12, 2], "spend": [500, 520, 900]}, index=["A", "B", "C"])

3,317 customers, churn rate 43.6%


train 2,487 / test 830


## Exercise 1: by hand

Standardise and min-max scale `orders` = 1, 2, 3, 4, 10 by hand (use the ÷n standard deviation). Which customer gets the most extreme value under each method? Check with scikit-learn.

In [2]:
# your code here

## Exercise 2: min-max and the wholesaler

Min-max scale the real `spend` column (fitted on `X_train`). What fraction of training customers end up below 0.01? Repeat after `np.log1p`. Which version would you give to KNN?

In [3]:
# your code here

## Exercise 3: the wrong way round

Fit a `StandardScaler` separately on `X_train` and on `X_test` (the mistake), train KNN (k = 15) on the scaled training data and score on the separately scaled test data. Compare with the right way. Then explain why scaling "live data on its own" would be even worse for a single customer.

In [4]:
# your code here

## Exercise 4: log, then scale

Build a version of `X_train` / `X_test` where `orders`, `spend` and `products` are log-transformed with `np.log1p` before standardising. Score KNN for k = 5, 15, 51 and compare with the lesson's table.

In [5]:
# your code here